# SENTINEL adapter -> GGUF q4_K_M on Kaggle (CPU)

Same steps as `export_gguf.py`, run on Kaggle so the local PC does not have to merge (~10 GB RAM).

Before **Save & Run All**:
1. Settings → Accelerator: **None** (CPU is enough, saves GPU quota) · Internet: **On**.
2. Add Input → Notebook output → `sentinel-qlora-r2` (the training notebook).

Output: `sentinel-qwen3.5-4b-r2-q4_K_M.gguf` + `Modelfile` (+ the adapter itself). On the M5:
`ollama create sentinel-qwen3.5-4b-r2 -f Modelfile` in the folder holding both files.

In [ ]:
BASE_REPO = "Qwen/Qwen3.5-4B"
BASE_REV = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"  # snapshot of D:\ai-models\Qwen3.5-4B (the one we trained on)
RUN = "sentinel-lora-r2"
NAME = "sentinel-qwen3.5-4b-r2"
LLAMA_CPP_REV = "1b29621bacc6a446bd523fc790316a24a06e886f"  # same as D:\ai-models\llama.cpp

In [ ]:
!pip install -q transformers==5.20.2 peft==0.21.2 accelerate==1.15.0 sentencepiece
import subprocess
subprocess.run(f"git init -q /tmp/llama.cpp && cd /tmp/llama.cpp && git fetch -q --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_CPP_REV} && git checkout -q FETCH_HEAD", shell=True, check=True)

In [ ]:
import shutil
from pathlib import Path

inp, working = Path("/kaggle/input"), Path("/kaggle/working")
# The final adapter if the run finished, else its latest checkpoint (r2 hit OOM at step 155, so checkpoint-150).
final = [p.parent for p in inp.rglob(f"{RUN}/adapter_model.safetensors")]
ckpts = sorted(inp.rglob(f"{RUN}/checkpoint-*/adapter_model.safetensors"), key=lambda p: int(p.parent.name.split("-")[1]))
adapter = final[0] if final else ckpts[-1].parent
print("adapter:", adapter)

keep = working / f"{RUN}-adapter"
keep.mkdir(exist_ok=True)
for name in ("adapter_config.json", "adapter_model.safetensors"):
    shutil.copy(adapter / name, keep)

In [ ]:
from huggingface_hub import snapshot_download

base = snapshot_download(BASE_REPO, revision=BASE_REV, local_dir="/tmp/Qwen3.5-4B")

In [ ]:
import json
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

merged = Path("/tmp/merged")
model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.bfloat16, low_cpu_mem_usage=True, device_map={"": "cpu"})
model = PeftModel.from_pretrained(model, str(adapter)).merge_and_unload()
model.save_pretrained(merged, safe_serialization=True, max_shard_size="2GB")
AutoTokenizer.from_pretrained(base).save_pretrained(merged)
del model
print(json.loads((merged / "config.json").read_text())["architectures"])

In [ ]:
import sys

f16 = Path("/tmp/model-f16.gguf")
subprocess.run([sys.executable, "/tmp/llama.cpp/convert_hf_to_gguf.py", str(merged), "--outfile", str(f16), "--outtype", "f16"], check=True)
shutil.rmtree(merged)

In [ ]:
subprocess.run("cd /tmp/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build build --target llama-quantize -j 4 > /dev/null", shell=True, check=True)
q4 = working / f"{NAME}-q4_K_M.gguf"
subprocess.run(["/tmp/llama.cpp/build/bin/llama-quantize", str(f16), str(q4), "Q4_K_M"], check=True)
f16.unlink()

# Same renderer/parser as qwen3.5:4b so tool calls and the chat format match the base model.
(working / "Modelfile").write_text(f"""FROM ./{q4.name}
TEMPLATE {{{{ .Prompt }}}}
RENDERER qwen3.5
PARSER qwen3.5
PARAMETER presence_penalty 1.5
PARAMETER temperature 1
PARAMETER top_k 20
PARAMETER top_p 0.95
""")
for p in sorted(working.rglob("*")):
    if p.is_file():
        print(p.relative_to(working), f"{p.stat().st_size / 2**20:.0f} MB")